In [10]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, GRU, Dense, Dropout,
    Bidirectional, Concatenate, BatchNormalization,
    GlobalMaxPooling1D, Attention
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

# ============================================
# 1. Load data (should have ~6000 samples)
# ============================================
df = pd.read_csv('/content/Data Ready for modeling (1).txt', low_memory=False)

cols = ['name', 'price', 'keyword', 'Brand Name', 'Flavor', 'Item Weight', 'Item Form']
df = df[cols]

# Clean price
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['price'])

# Clean text columns
for col in ['name', 'keyword', 'Brand Name', 'Flavor', 'Item Form']:
    df[col] = df[col].astype(str).str.strip().str.lower()

# Extract weight
def extract_weight_grams(weight_str):
    if pd.isna(weight_str):
        return None
    weight_str = str(weight_str)
    match = re.search(r'(\d+(?:\.\d+)?)\s*(Grams?|g|Kg|Kilograms?|ml|Milliliters?)', weight_str, re.IGNORECASE)
    if match:
        value = float(match.group(1))
        unit = match.group(2).lower()
        if 'kg' in unit or 'kilogram' in unit:
            return value * 1000
        elif 'g' in unit or 'gram' in unit:
            return value
        elif 'ml' in unit or 'milliliter' in unit:
            return value
    return None

df['actual_weight_gm'] = df['Item Weight'].apply(extract_weight_grams)
df['price_per_kg'] = df['price'] / df['actual_weight_gm'] * 1000

# Filter realistic prices (wider range for more data)
df = df[(df['price_per_kg'] > 5) & (df['price_per_kg'] < 800)]
df = df.dropna(subset=['price_per_kg', 'actual_weight_gm'])

print(f"📊 Total samples after filtering: {len(df)}")

# If less than 6000, we need to adjust strategy
if len(df) < 6000:
    print(f"⚠️ Warning: Only {len(df)} samples. Consider collecting more data or using different approach.")
    # Use less aggressive filtering
    df = pd.read_csv('/content/Data Ready for modeling (1).txt', low_memory=False)
    df = df[cols]
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    df = df.dropna(subset=['price'])
    for col in ['name', 'keyword', 'Brand Name', 'Flavor', 'Item Form']:
        df[col] = df[col].astype(str).str.strip().str.lower()
    df['actual_weight_gm'] = df['Item Weight'].apply(extract_weight_grams)
    df['price_per_kg'] = df['price'] / df['actual_weight_gm'] * 1000
    df = df[(df['price_per_kg'] > 0) & (df['price_per_kg'] < 1000)]
    df = df.dropna(subset=['price_per_kg', 'actual_weight_gm'])
    print(f"📊 After relaxed filtering: {len(df)} samples")

# Log transform target
df['price_per_kg_log'] = np.log1p(df['price_per_kg'])

# Extract pack size
df['pack_size'] = df['name'].str.extract(r'(\d+)\s*(?:piece|pcs|قطعة|pack|set|عدد)').astype(float)
df['pack_size'] = df['pack_size'].fillna(1)

# Create combined text input
df['text_input'] = (
    df['name'] + ' ' +
    df['keyword'] + ' ' +
    df['Brand Name'] + ' ' +
    df['Flavor'] + ' ' +
    df['Item Form']
)

print(f"✅ Final dataset size: {len(df)} samples")
print(f"💰 Price range: {df['price_per_kg'].min():.2f} - {df['price_per_kg'].max():.2f} EGP/kg")

# ============================================
# 2. Advanced Deep Learning Model for 6000 samples
# ============================================
MAX_VOCAB = 15000
MAX_SEQUENCE_LENGTH = 120
EMBEDDING_DIM = 128

# Tokenize
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(df['text_input'])
sequences = tokenizer.texts_to_sequences(df['text_input'])
X_text = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

# Numerical features
num_features = df[['actual_weight_gm', 'pack_size']].values
num_scaler = StandardScaler()
X_num = num_scaler.fit_transform(num_features)

# Target
y = df['price_per_kg_log'].values

# Split
X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, X_num, y, test_size=0.2, random_state=42, stratify=pd.qcut(y, q=4, labels=False) if len(y) >= 20 else None
)

print(f"\n📊 Training samples: {len(X_text_train)}")
print(f"📊 Test samples: {len(X_text_test)}")

# ============================================


📊 Total samples after filtering: 3360
⚠️ Warning: Only 3360 samples. Consider collecting more data or using different approach.
📊 After relaxed filtering: 3668 samples
✅ Final dataset size: 3668 samples
💰 Price range: 0.01 - 998.00 EGP/kg

📊 Training samples: 2934
📊 Test samples: 734


In [7]:
# ============================================
# Compile model (FIXED VERSION)
# ============================================

# Build model first
vocab_size = min(MAX_VOCAB, len(tokenizer.word_index) + 1)
model = build_advanced_hybrid_model(vocab_size, MAX_SEQUENCE_LENGTH, EMBEDDING_DIM, X_num.shape[1])

# Compile with fixed learning rate
model.compile(
    optimizer=Adam(learning_rate=0.001),  # ✅ Fixed - no schedule here
    loss='mse',
    metrics=['mae']
)

model.summary()

# Callbacks (ReduceLROnPlateau will adjust learning rate during training)
callbacks = [
    EarlyStopping(patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=8, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_deep_learning_model.h5', save_best_only=True, verbose=1)
]

# Train - this will work now
history = model.fit(
    [X_text_train, X_num_train], y_train,
    validation_split=0.15,
    epochs=80,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text_input          │ (None, 120)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 120, 128)  │    360,576 │ text_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 120)       │          0 │ text_input[0][0]  │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_4     │ (None, 120, 256)  │    263,168 │ embedding_2[0][0… │
│ (Bidirectional)     │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 120, 256)  │      1,024 │ bidirectional_4[… │
│ (BatchNormalizatio… │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_input           │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_5     │ (None, 120, 128)  │    123,648 │ batch_normalizat… │
│ (Bidirectional)     │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 64)        │        192 │ num_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 120, 128)  │        512 │ bidirectional_5[… │
│ (BatchNormalizatio… │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_9[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ batch_normalizat… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 64)        │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 256)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 32)        │      2,080 │ dropout_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 256)       │          0 │ concatenate_3[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 32)        │          0 │ dense_10[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 799,329 (3.05 MB)

 Trainable params: 798,177 (3.04 MB)

 Non-trainable params: 1,152 (4.50 KB)

...


Epoch 1: finished saving model to best_deep_learning_model.h5
78/78 ━━━━━━━━━━━━━━━━━━━━ 135s 1s/step - loss: 10.1817 - mae: 2.5164 - val_loss: 13.5427 - val_mae: 3.5499 - learning_rate: 0.0010
Epoch 2/80
78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 2.9963 - mae: 1.3800
Epoch 2: val_loss improved from 13.54271 to 5.24403, saving model to best_deep_learning_model.h5



Epoch 2: finished saving model to best_deep_learning_model.h5
78/78 ━━━━━━━━━━━━━━━━━━━━ 104s 1s/step - loss: 2.7478 - mae: 1.3283 - val_loss: 5.2440 - val_mae: 2.0616 - learning_rate: 0.0010
Epoch 3/80
78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 2.3936 - mae: 1.2397
Epoch 3: val_loss improved from 5.24403 to 1.66700, saving model to best_deep_learning_model.h5



Epoch 3: finished saving model to best_deep_learning_model.h5
78/78 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - loss: 2.2484 - mae: 1.2037 - val_loss: 1.6670 - val_mae: 1.0163 - learning_rate: 0.0010
Epoch 4/80
78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 1.9320 - mae: 1.1073
Epoch 4: val_loss improved from 1.66700 to 1.63340, saving model to best_deep_learning_model.h5



Epoch 4: finished saving model to best_deep_learning_model.h5
78/78 ━━━━━━━━━━━━━━━━━━━━ 143s 1s/step - loss: 1.9202 - mae: 1.1010 - val_loss: 1.6334 - val_mae: 0.8644 - learning_rate: 0.0010
Epoch 5/80
78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 1.6622 - mae: 1.0301
Epoch 5: val_loss did not improve from 1.63340
78/78 ━━━━━━━━━━━━━━━━━━━━ 117s 1s/step - loss: 1.6234 - mae: 1.0191 - val_loss: 2.2324 - val_mae: 0.9079 - learning_rate: 0.0010
Epoch 6/80
78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 1.6455 - mae: 1.0247
Epoch 6: val_loss did not improve from 1.63340
78/78 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - loss: 1.5754 - mae: 0.9974 - val_loss: 2.4777 - val_mae: 0.9024 - learning_rate: 0.0010
Epoch 7/80
78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 1.4884 - mae: 0.9794
Epoch 7: val_loss did not improve from 1.63340
78/78 ━━━━━━━━━━━━━━━━━━━━ 143s 1s/step - loss: 1.4060 - mae: 0.9484 - val_loss: 3.9398 - val_mae: 1.0465 - learning_rate: 0.0010
Epoch 8/80
65/78 ━━━━━━━━━━━━━━━━━━━━ 18s 

KeyboardInterrupt: 

In [14]:


def build_robust_hybrid_model(vocab_size=MAX_VOCAB,
                              embedding_dim=EMBEDDING_DIM,
                              max_length=MAX_SEQUENCE_LENGTH,
                              num_features=2):

    # Text input branch with strong regularization
    text_input = Input(shape=(max_length,), name='text_input')

    # Embedding layer
    x = Embedding(vocab_size, embedding_dim,
                  embeddings_initializer='he_normal')(text_input)

    # First LSTM with dropout and recurrent dropout
    x = Bidirectional(LSTM(64,
                           return_sequences=True,
                           dropout=0.3,
                           recurrent_dropout=0.3,
                           kernel_regularizer='l2'))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Second LSTM (smaller)
    x = Bidirectional(LSTM(32,
                           return_sequences=False,
                           dropout=0.3,
                           recurrent_dropout=0.3,
                           kernel_regularizer='l2'))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Numeric input branch
    num_input = Input(shape=(num_features,), name='numeric_input')
    y = Dense(32, activation='relu', kernel_regularizer='l2')(num_input)
    y = BatchNormalization()(y)
    y = Dropout(0.3)(y)
    y = Dense(16, activation='relu', kernel_regularizer='l2')(y)
    y = BatchNormalization()(y)
    y = Dropout(0.2)(y)

    # Combine branches
    combined = Concatenate()([x, y])

    # Final layers with strong regularization
    z = Dense(64, activation='relu', kernel_regularizer='l2')(combined)
    z = BatchNormalization()(z)
    z = Dropout(0.4)(z)

    z = Dense(32, activation='relu', kernel_regularizer='l2')(z)
    z = BatchNormalization()(z)
    z = Dropout(0.3)(z)

    z = Dense(16, activation='relu', kernel_regularizer='l2')(z)
    z = Dropout(0.2)(z)

    # Output layer
    output = Dense(1, name='output')(z)

    model = Model(inputs=[text_input, num_input], outputs=output)

    return model


def get_callbacks():
    # Early stopping with patience
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=15,  # Wait 15 epochs before stopping
        restore_best_weights=True,
        verbose=1,
        mode='min'
    )

    # Reduce learning rate when plateau
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1,
        mode='min'
    )

    # Save only the best model
    checkpoint = ModelCheckpoint(
        'best_hybrid_model.keras',  # Use .keras format instead of .h5
        monitor='val_loss',
        save_best_only=True,
        verbose=1,
        mode='min'
    )

    # Custom callback to monitor overfitting
    class OverfittingMonitor(tf.keras.callbacks.Callback):
        def __init__(self, threshold=1.5):
            super().__init__()
            self.threshold = threshold

        def on_epoch_end(self, epoch, logs=None):
            val_loss = logs.get('val_loss')
            train_loss = logs.get('loss')

            if val_loss and train_loss:
                ratio = val_loss / train_loss
                if ratio > self.threshold:
                    print(f"\n WARNING: Overfitting detected! val_loss/train_loss = {ratio:.3f}")
                    print(f"   Consider stopping training or increasing regularization")

                # Stop if extreme overfitting
                if ratio > 2.5:
                    print(f"\nExtreme overfitting detected! Stopping training...")
                    self.model.stop_training = True

    overfit_monitor = OverfittingMonitor(threshold=1.5)

    return [early_stop, reduce_lr, checkpoint, overfit_monitor]



# Build model
model = build_robust_hybrid_model(
    vocab_size=MAX_VOCAB,
    embedding_dim=EMBEDDING_DIM,
    max_length=MAX_SEQUENCE_LENGTH,
    num_features=2
)

# Use AdamW (Adam with weight decay) for better regularization
from tensorflow.keras.optimizers import AdamW

initial_lr = 0.001
optimizer = AdamW(
    learning_rate=initial_lr,
    weight_decay=0.001,  # Important for regularization!
    beta_1=0.9,
    beta_2=0.999
)

model.compile(
    optimizer=optimizer,
    loss='huber',  # More robust to outliers than MSE
    metrics=['mae']
)

# Model summary
model.summary()


#  Data Augmentation for Text (Optional but helpful)

def augment_text_data(X_text, X_num, y, augmentation_factor=2):
    """Simple data augmentation by adding noise to embeddings"""
    X_text_aug = []
    X_num_aug = []
    y_aug = []

    for i in range(len(y)):
        # Original
        X_text_aug.append(X_text[i])
        X_num_aug.append(X_num[i])
        y_aug.append(y[i])

        # Augmented versions
        for _ in range(augmentation_factor - 1):
            # Add small noise to numeric features
            noise_num = X_num[i] + np.random.normal(0, 0.01, X_num[i].shape)
            X_num_aug.append(noise_num)
            X_text_aug.append(X_text[i])
            y_aug.append(y[i])

    return np.array(X_text_aug), np.array(X_num_aug), np.array(y_aug)

# Only augment if dataset is small
if len(X_text_train) < 4000:
    print("\nAugmenting training data...")
    X_text_train, X_num_train, y_train = augment_text_data(
        X_text_train, X_num_train, y_train, augmentation_factor=2
    )
    print(f"After augmentation: {len(X_text_train)} samples")



from sklearn.model_selection import KFold

def train_with_cross_validation(X_text, X_num, y, n_splits=5):
    """Train using cross-validation to detect overfitting"""

    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_scores = []
    best_models = []

    for fold, (train_idx, val_idx) in enumerate(kfold.split(X_text)):
        print(f"\n{'='*50}")
        print(f"FOLD {fold+1}/{n_splits}")
        print(f"{'='*50}")

        # Split data
        X_text_tr, X_text_val = X_text[train_idx], X_text[val_idx]
        X_num_tr, X_num_val = X_num[train_idx], X_num[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        # Build model
        model_fold = build_robust_hybrid_model()
        model_fold.compile(
            optimizer=AdamW(learning_rate=0.001, weight_decay=0.001),
            loss='huber',
            metrics=['mae']
        )

        # Train
        history = model_fold.fit(
            [X_text_tr, X_num_tr], y_tr,
            validation_data=([X_text_val, X_num_val], y_val),
            epochs=50,
            batch_size=32,
            callbacks=get_callbacks(),
            verbose=1
        )

        # Evaluate
        val_mae = min(history.history['val_mae'])
        fold_scores.append(val_mae)
        best_models.append(model_fold)

        print(f"Fold {fold+1} best val_mae: {val_mae:.4f}")

    print(f"\nCross-validation results:")
    print(f"   Mean val_mae: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")

    # Return best model from all folds
    best_idx = np.argmin(fold_scores)
    return best_models[best_idx], fold_scores

# Choose training method based on dataset size
if len(X_text_train) < 2000:
    print("\n🔄 Using Cross-Validation for small dataset...")
    model, cv_scores = train_with_cross_validation(X_text_train, X_num_train, y_train)
else:
    print("\ Using standard training...")
    history = model.fit(
        [X_text_train, X_num_train], y_train,
        validation_split=0.2,
        epochs=50,
        batch_size=32,
        callbacks=get_callbacks(),
        verbose=1
    )


from tensorflow.keras.models import load_model

try:
    best_model = load_model('best_hybrid_model.keras', compile=False)
    best_model.compile(optimizer=optimizer, loss='huber', metrics=['mae'])
except:
    best_model = model

# Make predictions
y_pred_log = best_model.predict([X_text_test, X_num_test])
y_pred = np.expm1(y_pred_log.flatten())  # Reverse log transform
y_test_original = np.expm1(y_test)

# Calculate metrics
mae = mean_absolute_error(y_test_original, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_original, y_pred))
r2 = r2_score(y_test_original, y_pred)
mape = np.mean(np.abs((y_test_original - y_pred) / y_test_original)) * 100

print("\n" + "="*50)
print("📈 FINAL MODEL PERFORMANCE")
print("="*50)
print(f" MAE:  {mae:.2f} EGP/kg")
print(f" RMSE: {rmse:.2f} EGP/kg")
print(f" R²:   {r2:.4f}")
print(f" MAPE: {mape:.2f}%")


# Check for overfitting in final model
train_pred_log = best_model.predict([X_text_train, X_num_train])
train_pred = np.expm1(train_pred_log.flatten())
y_train_original = np.expm1(y_train)

train_mae = mean_absolute_error(y_train_original, train_pred)
test_mae = mae

print(f" Overfitting Check:")
print(f"   Training MAE: {train_mae:.2f}")
print(f"   Testing MAE:  {test_mae:.2f}")
print(f"   Difference:   {abs(train_mae - test_mae):.2f}")

if abs(train_mae - test_mae) > test_mae * 0.3:
    print("    Warning: Possible overfitting detected!")
else:
    print("Good: No significant overfitting!")

# ============================================
# 9. Save model and preprocessing objects
# ============================================

import joblib

# Save model in new format
best_model.save('final_hybrid_model.keras')

# Save preprocessing objects
joblib.dump(tokenizer, 'tokenizer.pkl')
joblib.dump(num_scaler, 'num_scaler.pkl')

print("\n Model and preprocessing objects saved successfully!")

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text_input          │ (None, 120)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 120, 128)  │  1,920,000 │ text_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric_input       │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_8     │ (None, 120, 128)  │     98,816 │ embedding_4[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 32)        │         96 │ numeric_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 120, 128)  │        512 │ bidirectional_8[… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32)        │        128 │ dense_19[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_22          │ (None, 120, 128)  │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_24          │ (None, 32)        │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_9     │ (None, 64)        │     41,216 │ dropout_22[0][0]  │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 16)        │        528 │ dropout_24[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ bidirectional_9[… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16)        │         64 │ dense_20[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_23          │ (None, 64)        │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_25          │ (None, 16)        │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, 80)        │          0 │ dropout_23[0][0], │
│ (Concatenate)       │                   │            │ dropout_25[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 64)        │      5,184 │ concatenate_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_21[0][0]  

 Total params: 2,069,809 (7.90 MB)

 Trainable params: 2,069,137 (7.89 MB)

 Non-trainable params: 672 (2.62 KB)

\ Using standard training...
Epoch 1/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 609ms/step - loss: 9.8023 - mae: 4.3330
Epoch 1: val_loss improved from None to 5.26046, saving model to best_hybrid_model.keras

Epoch 1: finished saving model to best_hybrid_model.keras
147/147 ━━━━━━━━━━━━━━━━━━━━ 119s 660ms/step - loss: 8.0632 - mae: 3.5067 - val_loss: 5.2605 - val_mae: 1.7428 - learning_rate: 0.0010
Epoch 2/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 605ms/step - loss: 5.3567 - mae: 1.9940
Epoch 2: val_loss improved from 5.26046 to 3.90140, saving model to best_hybrid_model.keras

Epoch 2: finished saving model to best_hybrid_model.keras
147/147 ━━━━━━━━━━━━━━━━━━━━ 94s 642ms/step - loss: 5.0946 - mae: 1.8759 - val_loss: 3.9014 - val_mae: 0.9198 - learning_rate: 0.0010
Epoch 3/50
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 604ms/step - loss: 4.5028 - mae: 1.6812
Epoch 3: val_loss improved from 3.90140 to 3.56824, saving model to best_hybrid_model.keras

Epoch 3: finished saving model to best_hybrid_model.keras
147

In [22]:

tolerance = 0.2  # 20% تسامح
accurate_predictions = np.abs((y_test_original - y_pred) / y_test_original) <= tolerance
accuracy_like = np.mean(accurate_predictions) * 100

print(f" {accuracy_like:.2f}%")

 33.65%
